In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
from torchvision import transforms

transform = transforms.Compose([

    # tamaño de EfficientNet
    transforms.Resize((224,224)),

    # espejo horizontal
    transforms.RandomHorizontalFlip(p=0.5),

    # rotaciones más agresivas
    transforms.RandomRotation(25),

    # cambios de iluminación y color
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.1
    ),

    # zoom + desplazamiento
    transforms.RandomAffine(
        degrees=0,
        translate=(0.15, 0.15),
        scale=(0.8, 1.2)
    ),

    # perspectiva tipo cámara real
    transforms.RandomPerspective(
        distortion_scale=0.25,
        p=0.4
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    ),

    # oclusiones parciales (mano, sombras, objetos)
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02,0.12)
    )
])

In [3]:
import os
import json
from torchvision import datasets
from torch.utils.data import DataLoader

DATASET_PATH = "../data/food-101/images"

# =========================
# CARGAR DATASET
# =========================
dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

# clases reales detectadas automáticamente
CLASSES = dataset.classes

# guardar clases para la app
os.makedirs("../models", exist_ok=True)

with open("../models/classes.json", "w") as f:
    json.dump(CLASSES, f)

# =========================
# DATALOADER
# =========================
loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total imágenes:", len(dataset))
print("Total clases:", len(CLASSES))
print("Clases reales:", CLASSES)

Total imágenes: 19545
Total clases: 21
Clases reales: ['apple', 'banana', 'ceviche', 'chicken_wings', 'coca_cola', 'coffee', 'french_fries', 'fried_rice', 'hamburger', 'ice_cream', 'lemon', 'mango', 'monster_energy', 'nachos', 'pizza', 'ramen', 'spaghetti_bolognese', 'steak', 'tacos', 'water_bottle', 'watermelon']


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

OLD_MODEL_PATH = "../models/modelo_efficientnet.pth"

# =========================
# DETECTAR CLASES DEL MODELO ANTERIOR
# =========================
checkpoint = torch.load(
    OLD_MODEL_PATH,
    map_location="cpu"
)

old_classes = checkpoint[
    "classifier.1.weight"
].shape[0]

# =========================
# MODELO NUEVO (N clases)
# =========================
model = efficientnet_b0(weights=None)

new_num_classes = len(CLASSES)

# nueva capa final
new_classifier = nn.Linear(
    1280,
    new_num_classes
)

# =========================
# CARGAR MODELO ANTERIOR
# =========================
old_model = efficientnet_b0(weights=None)

old_model.classifier[1] = nn.Linear(
    1280,
    old_classes
)

old_model.load_state_dict(checkpoint)

# =========================
# COPIAR CONOCIMIENTO VIEJO
# =========================
with torch.no_grad():

    # inicializar nuevas neuronas
    nn.init.xavier_uniform_(
        new_classifier.weight
    )

    # copiar clases antiguas
    new_classifier.weight[
        :old_classes
    ] = old_model.classifier[1].weight

    new_classifier.bias[
        :old_classes
    ] = old_model.classifier[1].bias


# reemplazar classifier
model.classifier[1] = new_classifier

# copiar backbone completo
model.features.load_state_dict(
    old_model.features.state_dict()
)

# =========================
# FASE 1
# congelar backbone
# =========================
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True


print("Modelo incremental listo")
print("Clases antiguas:", old_classes)
print("Clases nuevas:", len(CLASSES))

C:\Users\brayn\AppData\Local\Temp\ipykernel_12000\1926508640.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(OLD_MODEL_PATH, map_location="cpu")


RuntimeError: Error(s) in loading state_dict for EfficientNet:
	size mismatch for classifier.1.weight: copying a param with shape torch.Size([21, 1280]) from checkpoint, the shape in current model is torch.Size([10, 1280]).
	size mismatch for classifier.1.bias: copying a param with shape torch.Size([21]) from checkpoint, the shape in current model is torch.Size([10]).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Usando:", device)

criterion = nn.CrossEntropyLoss()


# =========================
# FASE 1
# SOLO CLASIFICADOR
# =========================
print("Fase 1: Adaptando nuevas clases")

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

EPOCHS = 8

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"[F1] Epoch {epoch+1}/{EPOCHS}, "
        f"Loss: {total_loss:.4f}"
    )


# =========================
# FASE 2
# FINE TUNING SUAVE
# =========================
print("Fase 2: Fine-tuning global")

for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    model.parameters(),
    lr=0.00003
)

EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"[F2] Epoch {epoch+1}/{EPOCHS}, "
        f"Loss: {total_loss:.4f}"
    )


# =========================
# GUARDAR
# =========================
torch.save(
    model.state_dict(),
    "../models/modelo_efficientnet.pth"
)

print("Modelo actualizado guardado")

Usando: cuda
Fase 1: Adaptando nuevas clases
[F1] Epoch 1/5, Loss: 424.2204
[F1] Epoch 2/5, Loss: 82.0397
[F1] Epoch 3/5, Loss: 63.8700
[F1] Epoch 4/5, Loss: 56.0853
[F1] Epoch 5/5, Loss: 50.6767
Fase 2: Fine-tuning global
[F2] Epoch 1/10, Loss: 43.8550
[F2] Epoch 2/10, Loss: 31.9424
[F2] Epoch 3/10, Loss: 25.1844
[F2] Epoch 4/10, Loss: 21.1196
[F2] Epoch 5/10, Loss: 16.5941
[F2] Epoch 6/10, Loss: 14.2061
[F2] Epoch 7/10, Loss: 12.7126
[F2] Epoch 8/10, Loss: 11.0411
[F2] Epoch 9/10, Loss: 9.7215
[F2] Epoch 10/10, Loss: 9.1401
Modelo actualizado guardado


In [ ]:
torch.save(model.state_dict(), "../models/modelo_efficientnet.pth")
print("Modelo guardado correctamente")

Modelo guardado correctamente


In [ ]:
import json

ruta = r"C:\Users\brayn\Desktop\DOCS DEV\Clasificador de Alimentos\models\classes.json"

with open(ruta, "w") as f:
    json.dump(CLASSES, f)